In [1]:
from util import import_ragas_custom, load_env_variables_from_all_env_files

import_ragas_custom('ragas_custom_2')

Arquivos copiados com sucesso!


In [2]:
load_env_variables_from_all_env_files()

In [3]:
import os
import asyncio
import nest_asyncio

from llama_index.core import SimpleDirectoryReader
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import default_query_distribution
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding, OpenAIEmbeddingModelType
from ragas.run_config import RunConfig
from ragas.llms import LlamaIndexLLMWrapper
from ragas.embeddings import LlamaIndexEmbeddingsWrapper
from ragas.testset.transforms import default_transforms
from ragas.testset.transforms.engine import Parallel
from ragas.prompt import PromptMixin

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jmess\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [4]:
nest_asyncio.apply()

In [5]:
DATA = 'data'
LANGUAGE = "portuguese"
TIMEOUT = 2400.0
CACHE_DIR = 'cache_2'
AMOUNT_TESTS = 128
MODEL = 'gpt-4o-mini-2024-07-18'

In [6]:
run_config = RunConfig()

In [7]:
llm = LlamaIndexLLMWrapper(OpenAI(model=MODEL), run_config=run_config)
embedding = LlamaIndexEmbeddingsWrapper(OpenAIEmbedding(model=OpenAIEmbeddingModelType.TEXT_EMBED_3_SMALL), run_config=run_config)
generator = TestsetGenerator(llm)

In [8]:
query_distribution = default_query_distribution(llm)

transforms = default_transforms(
                llm=llm,
                embedding_model=embedding,
            )

In [9]:
list_transforms = [i for i, _ in query_distribution]

for i in transforms:
    if isinstance(i, Parallel):
        list_transforms.extend(i.transformations)
    else:
        list_transforms.append(i)

list_transforms = [i for i in list_transforms if isinstance(i, PromptMixin)]

for query in list_transforms:
    path = os.path.join(CACHE_DIR, query.__class__.__name__)
    if not os.path.exists(path):
        os.makedirs(path)

    try:
        prompts = query.load_prompts(path, LANGUAGE)
        query.set_prompts(**prompts)
    except Exception:
        prompts = asyncio.run(query.adapt_prompts(LANGUAGE, None, True, True))
        query.set_prompts(**prompts)
        query.save_prompts(path)
        prompts = query.load_prompts(path, LANGUAGE)
        query.set_prompts(**prompts)


In [10]:
documents = SimpleDirectoryReader(DATA).load_data()

In [11]:
testset = generator.generate_with_llamaindex_docs(documents, AMOUNT_TESTS, query_distribution=query_distribution,
                                                  with_debugging_logs=True, run_config=run_config, 
                                                  transforms_llm=llm,
                                                  transforms_embedding_model=embedding,
                                                  transforms=transforms)

Applying [SummaryExtractor, HeadlinesExtractor]:   0%|          | 0/114 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/57 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/57 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, KeyphrasesExtractor, TitleExtractor]:   0%|          | 0/762 [00:00<?, ?it/s]

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

...

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating common_concepts:   0%|          | 0/2 [00:00<?, ?it/s]

Generating common themes:   0%|          | 0/18 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/132 [00:00<?, ?it/s]

Retrying llama_index.llms.openai.base.OpenAI._achat in 0.6599005975495235 seconds as it raised RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-JYT7WTlT5eXr5JOV54kIyrgW on tokens per min (TPM): Limit 200000, Used 199440, Requested 753. Please try again in 57ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}.
Retrying llama_index.llms.openai.base.OpenAI._achat in 0.34564886160716246 seconds as it raised RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-JYT7WTlT5eXr5JOV54kIyrgW on tokens per min (TPM): Limit 200000, Used 199585, Requested 510. Please try again in 28ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}.
Retrying llama_index.llms.openai.base.OpenAI._achat in 1.7255475898381412 s

In [12]:
testset.to_jsonl('testset_openai_4omini.jsonl')